# 膜分離器の必要性分析 — Dist3 単体 vs 膜+Dist3 比較

## 目的

PDH プロセスにおいて、C3H6/C3H8 分離のために **膜分離器 (Membrane) を Dist3 (C3 スプリッタ) の前段に導入する必要があるか** を定量評価する。

## 比較対象

- **Case A: 膜あり** (現行設計)  
  Dist2 塔底 → **Mem** (96% C3H6 に予備濃縮) → **Dist3** (99.5 wt% 製品)

- **Case B: 膜なし**  
  Dist2 塔底 → **Dist3 単体** で 37% → 99.5 wt% まで分離

## 物理的論点

- α (C3H6/C3H8 揮発度比) は 17 bar で **~1.07** (極小)。古典的に分離困難系。
- 低 α 系では Fenske の N_min が組成に対し急激に増加する。
- 膜 (Hua et al. 2024 の高選択性膜: α=90) で予備濃縮することで Dist3 の負担を激減できるはず。

## 参照ケース

本 notebook では `exp1 #282` (main_20260524_001733 BO best、TAC=978 億円) の Dist2 塔底組成を基準ストリームとして使用する。

## 1. セットアップ

In [ ]:
import sys, os
# プロジェクトルートを sys.path に追加 (modelling/project/membrane_vs_dist3 → root は 3 階層上)
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 日本語表示
plt.rcParams['font.family'] = ['MS Gothic', 'Yu Gothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# プロジェクトモジュール
from flowsheet import FlowsheetDesignVars  # 循環 import 回避のため先に
from src.distillation_core import (
    simulate_distillation_column, DistDesignVars, DistFixedParams
)
from stream.stream import ProcessStream
from units.separators.membrane.membrane_system import (
    MemDesignVars, MemFeedStream, MemFixedParams, simulate_membrane_system
)
from src.cost_parameters import (
    LP_STEAM_JPY_PER_GJ, COOLING_WATER_JPY_PER_GJ,
    OPERATING_HOURS_PER_YEAR, DEPRECIATION_YEARS, ELECTRICITY_JPY_PER_KWH,
)

MW_C3H6 = 42.08  # kg/kmol
MW_C3H8 = 44.10

print('プロジェクトルート:', _ROOT)
print(f'運転時間: {OPERATING_HOURS_PER_YEAR} h/年, 償却: {DEPRECIATION_YEARS} 年')

## 2. 基準条件 (exp1 #282 の Dist2 塔底)

Dist2 塔底からの C3 リッチストリームを膜および Dist3 の共通フィードとする。

In [ ]:
# Dist2 塔底 (exp1 #282 出力より)
DIST2_BOT = {
    'F_C3H8': 3848.3,   # kmol/h
    'F_C3H6': 2279.2,
    'F_C2H6': 21.6,     # 微量 (無視可)
    'T': 275.43,        # K (約 2°C)
    'P': 5.58e5,        # Pa (5.58 bar)
}
TARGET_PURITY_WT = 0.995    # spec: 99.5 wt% C3H6
P_DIST3 = 17.43e5            # Pa (Mem 透過側 P_dist と同期)

F_total = DIST2_BOT['F_C3H8'] + DIST2_BOT['F_C3H6']
x_C3H6_feed = DIST2_BOT['F_C3H6'] / F_total
print(f'Dist2 bot 組成: C3H6 = {x_C3H6_feed*100:.2f} mol%, 総流量 = {F_total:.1f} kmol/h')
print(f'目標 C3H6 純度: {TARGET_PURITY_WT*100:.1f} wt%')
print(f'P_dist3 (= Mem 透過側 P_dist): {P_DIST3/1e5:.2f} bar')

## 3. ヘルパー関数: Dist3 シミュレーション

In [ ]:
def simulate_dist3(F_C3H6, F_C3H8, P_col=P_DIST3,
                    N_stages=200, reflux_ratio=12.0, T_feed=320.0,
                    rec_LK=0.99, rec_HK=0.99):
    """Dist3 (C3H6=LK, C3H8=HK) を FUG で評価。
    
    Returns dict with key results.
    """
    feed = ProcessStream(
        F_in={'A': F_C3H8, 'B': F_C3H6},   # 'A'=C3H8, 'B'=C3H6
        T_in=T_feed, P_in=P_col,
    )
    design = DistDesignVars(
        P_col=P_col, N_stages=N_stages,
        N_feed=N_stages // 2,
        reflux_ratio=reflux_ratio,
        LK='B', HK='A',
        recovery_LK_top=rec_LK,
        recovery_HK_bot=rec_HK,
        K_method='pr', q=1.0,
        partial_condenser=False,
        solver_method='fug',
    )
    res = simulate_distillation_column(design, feed)
    eq = res.equipment
    # purity (top) wt%
    top = res.top.F_in
    top_mass = top.get('A', 0)*MW_C3H8 + top.get('B', 0)*MW_C3H6
    purity_wt = top.get('B', 0)*MW_C3H6 / top_mass if top_mass > 0 else 0
    # OPEX 概算 (リボイラ蒸気 + コンデンサ冷却水)
    opex_reb = eq.Q_reb * 3.6e-3 * OPERATING_HOURS_PER_YEAR * eq.reb_utility_jpy_per_GJ / 1e8
    opex_cond = eq.Q_cond * 3.6e-3 * OPERATING_HOURS_PER_YEAR * eq.cond_utility_jpy_per_GJ / 1e8
    capex_annual = eq.CAPEX / DEPRECIATION_YEARS
    return {
        'N_stages': N_stages, 'R': reflux_ratio,
        'N_min': eq.N_min, 'R_min': eq.R_min,
        'feasible': eq.feasible,
        'N_needed': eq.N_needed,
        'D_col_m': eq.D_col, 'H_col_m': eq.H_col,
        'CAPEX_total': eq.CAPEX,
        'CAPEX_vessel': eq.CAPEX_vessel,
        'CAPEX_cond': eq.CAPEX_cond,
        'CAPEX_reb': eq.CAPEX_reb,
        'Q_reb_kW': eq.Q_reb, 'Q_cond_kW': eq.Q_cond,
        'OPEX_reb': opex_reb, 'OPEX_cond': opex_cond,
        'OPEX_total': opex_reb + opex_cond,
        'CAPEX_annual': capex_annual,
        'TAC_dist3': capex_annual + opex_reb + opex_cond,
        'purity_wt': purity_wt,
        'F_C3H6_top': top.get('B', 0),
        'F_C3H8_top': top.get('A', 0),
        'T_top': eq.T_top, 'T_bot': eq.T_bot,
    }

print('ヘルパー関数定義完了')

## 4. Case A: 膜あり (Mem → Dist3)

exp1 #282 の Mem パラメータで simulate し、permeate を Dist3 のフィードとする。

In [ ]:
# 4.1 Mem シミュレーション
mem_design = MemDesignVars(
    P_H=8.75e5, P_L=1.0e5,
    A_mem=1.45e5,
    P_dist=P_DIST3,
)
mem_feed = MemFeedStream(
    F_C3H6=DIST2_BOT['F_C3H6'],
    F_C3H8=DIST2_BOT['F_C3H8'],
    T_in=DIST2_BOT['T'],
    P_in=DIST2_BOT['P'],
)
mem_result = simulate_membrane_system(mem_design, mem_feed, MemFixedParams())

perm_total = mem_result.product.F_C3H6 + mem_result.product.F_C3H8
perm_x_C3H6 = mem_result.product.F_C3H6 / perm_total if perm_total > 0 else 0
ret_total = mem_result.retentate.F_C3H6 + mem_result.retentate.F_C3H8
stage_cut = perm_total / (perm_total + ret_total)

print(f'=== Mem シミュレーション結果 ===')
print(f'透過 (perm):    C3H6={mem_result.product.F_C3H6:7.1f} kmol/h, C3H8={mem_result.product.F_C3H8:6.1f}, 計 {perm_total:7.1f}')
print(f'保留 (ret):     C3H6={mem_result.retentate.F_C3H6:7.1f} kmol/h, C3H8={mem_result.retentate.F_C3H8:6.1f}, 計 {ret_total:7.1f}')
print(f'Stage cut:      {stage_cut*100:.1f}%')
print(f'透過 C3H6 mol%: {perm_x_C3H6*100:.2f}%')
print(f'保留 C3H6 mol%: {mem_result.retentate.F_C3H6/ret_total*100:.2f}%')
print(f'\nMem CAPEX 内訳 [億円]:')
print(f'  気化器:        {mem_result.equipment.CAPEX_vap:.2f}')
print(f'  F 圧縮機:      {mem_result.equipment.CAPEX_comp_feed:.2f}')
print(f'  膜本体:        {mem_result.equipment.CAPEX_mem:.2f}')
print(f'  P 圧縮機:      {mem_result.equipment.CAPEX_comp_prod:.2f}')
print(f'  冷却器:        {mem_result.equipment.CAPEX_cond:.2f}')
MEM_CAPEX_TOTAL = (mem_result.equipment.CAPEX_vap + mem_result.equipment.CAPEX_comp_feed
                    + mem_result.equipment.CAPEX_mem + mem_result.equipment.CAPEX_comp_prod
                    + mem_result.equipment.CAPEX_cond)
print(f'  小計:          {MEM_CAPEX_TOTAL:.2f} 億円')

In [ ]:
# 4.2 Mem OPEX 概算 (圧縮機電力 + 気化器蒸気 + 冷却器水)
MEM_OPEX_elec = (mem_result.equipment.W_feed_kW + mem_result.equipment.W_prod_kW) * ELECTRICITY_JPY_PER_KWH * OPERATING_HOURS_PER_YEAR / 1e8
MEM_OPEX_steam = mem_result.equipment.Q_vap_kW * 3.6e-3 * OPERATING_HOURS_PER_YEAR * LP_STEAM_JPY_PER_GJ / 1e8
MEM_OPEX_water = mem_result.equipment.Q_cond_kW * 3.6e-3 * OPERATING_HOURS_PER_YEAR * COOLING_WATER_JPY_PER_GJ / 1e8
MEM_OPEX_TOTAL = MEM_OPEX_elec + MEM_OPEX_steam + MEM_OPEX_water
MEM_TAC_ANNUAL = MEM_CAPEX_TOTAL / DEPRECIATION_YEARS + MEM_OPEX_TOTAL

print(f'Mem OPEX 内訳 [億円/年]:')
print(f'  圧縮機電力:    {MEM_OPEX_elec:.2f}  (W_feed+W_prod = {mem_result.equipment.W_feed_kW+mem_result.equipment.W_prod_kW:.0f} kW)')
print(f'  気化器蒸気:    {MEM_OPEX_steam:.2f}  (Q_vap = {mem_result.equipment.Q_vap_kW:.0f} kW)')
print(f'  冷却器水:      {MEM_OPEX_water:.2f}  (Q_cond = {mem_result.equipment.Q_cond_kW:.0f} kW)')
print(f'  小計:          {MEM_OPEX_TOTAL:.2f} 億円/年')
print(f'\nMem TAC (CAPEX/{DEPRECIATION_YEARS}年 + OPEX): {MEM_TAC_ANNUAL:.2f} 億円/年')

In [ ]:
# 4.3 Dist3 を Mem 透過側を入力としてシミュレート
# exp1 #282: N=121, R=10.105, rec_LK=0.9859, rec_HK=0.9838
caseA = simulate_dist3(
    F_C3H6=mem_result.product.F_C3H6,
    F_C3H8=mem_result.product.F_C3H8,
    P_col=P_DIST3,
    N_stages=121, reflux_ratio=10.105,
    T_feed=mem_result.product.T_out,
    rec_LK=0.9859, rec_HK=0.9838,
)
print(f'=== Case A: Dist3 (Mem 透過側入力) ===')
print(f'Feed C3H6 mol%: {perm_x_C3H6*100:.2f}%')
print(f'N_stages = {caseA["N_stages"]}, R = {caseA["R"]:.2f}, N_min = {caseA["N_min"]:.1f}, R_min = {caseA["R_min"]:.2f}')
print(f'feasible: {caseA["feasible"]}')
print(f'塔径 = {caseA["D_col_m"]:.2f} m, 塔高 = {caseA["H_col_m"]:.1f} m')
print(f'C3H6 純度 (top): {caseA["purity_wt"]*100:.4f} wt% (target 99.5 wt%)')
print(f'CAPEX 合計 = {caseA["CAPEX_total"]:.2f} 億円 (vessel {caseA["CAPEX_vessel"]:.2f} + cond {caseA["CAPEX_cond"]:.2f} + reb {caseA["CAPEX_reb"]:.2f})')
print(f'OPEX = {caseA["OPEX_total"]:.2f} 億円/年 (reb {caseA["OPEX_reb"]:.2f} + cond {caseA["OPEX_cond"]:.2f})')
print(f'Q_reb = {caseA["Q_reb_kW"]:.0f} kW, Q_cond = {caseA["Q_cond_kW"]:.0f} kW')
print(f'Dist3 TAC = {caseA["TAC_dist3"]:.2f} 億円/年')
print(f'\n*** Case A 合計 TAC (Mem+Dist3) = {MEM_TAC_ANNUAL + caseA["TAC_dist3"]:.2f} 億円/年 ***')

## 5. Case B: 膜なし (Dist3 単体で Dist2 塔底を直接処理)

37 mol% C3H6 ストリームを 99.5 wt% まで分離する場合、Dist3 の段数・還流比はどこまで必要か?

**重要点**: rec_LK=0.99, rec_HK=0.99 (Case A と同 spec) では、Top に漏れる C3H8 が 1% × 3848 = 38 kmol/h となり、これが C3H6 純度を 98.24 wt% に頭打ちにする。
spec 99.5 wt% を達成するには **rec_HK ≥ 0.999** が必須 → N_min が爆発する。

In [ ]:
# 5.1 様々な N, R, recovery で Dist3 単体を試行
caseB_attempts = []
test_params = [
    # (N, R, rec_LK, rec_HK, 注釈)
    (121, 10.1, 0.99, 0.99,   '同 Case A 規模'),
    (300, 20,   0.99, 0.99,   'デフォルト rec, N=R↑'),
    (300, 30,   0.99, 0.99,   'R↑↑ (rec 据置)'),
    (300, 30,   0.99, 0.999,  'rec_HK↑ で純度確保狙い'),
    (200, 25,   0.99, 0.999,  'N=200, rec_HK↑'),
    (500, 25,   0.99, 0.999,  'N=500, rec_HK↑'),
    (300, 30,   0.999, 0.999, '両 rec ↑↑'),
]
for N, R, rL, rH, note in test_params:
    try:
        r = simulate_dist3(
            F_C3H6=DIST2_BOT['F_C3H6'],
            F_C3H8=DIST2_BOT['F_C3H8'],
            P_col=P_DIST3,
            N_stages=N, reflux_ratio=R,
            T_feed=DIST2_BOT['T'] + 50,
            rec_LK=rL, rec_HK=rH,
        )
        purity_meets_spec = r['purity_wt'] >= 0.995
        caseB_attempts.append({
            'note': note,
            'N': N, 'R': R, 'rec_LK': rL, 'rec_HK': rH,
            'N_min': r['N_min'], 'R_min': r['R_min'],
            'feasible': r['feasible'],
            'purity_wt': r['purity_wt'],
            'spec_met': purity_meets_spec,
            'CAPEX': r['CAPEX_total'],
            'TAC': r['TAC_dist3'],
            'Q_reb_MW': r['Q_reb_kW']/1000,
            'D_col': r['D_col_m'], 'H_col': r['H_col_m'],
        })
    except Exception as e:
        caseB_attempts.append({'note': note, 'N': N, 'R': R, 'error': str(e)[:50]})

df_caseB = pd.DataFrame(caseB_attempts)
df_caseB

In [ ]:
# 5.2 Case B 最善ケース (spec 99.5 wt% 達成 + 最安 TAC) と Case A 比較
df_caseB_meet = df_caseB[df_caseB.get('spec_met', False) == True] if 'spec_met' in df_caseB.columns else df_caseB.iloc[0:0]

if len(df_caseB_meet) > 0:
    best_B = df_caseB_meet.loc[df_caseB_meet['TAC'].idxmin()]
    print(f'=== Case B 最善 (spec 99.5 wt% 達成) ===')
    print(f'構成: N = {best_B["N"]}, R = {best_B["R"]}, rec = ({best_B["rec_LK"]:.4f}, {best_B["rec_HK"]:.4f})')
    print(f'塔径 = {best_B["D_col"]:.2f}m, 塔高 = {best_B["H_col"]:.1f}m')
    print(f'純度 = {best_B["purity_wt"]*100:.3f} wt%')
    print(f'CAPEX = {best_B["CAPEX"]:.2f} 億円, Q_reb = {best_B["Q_reb_MW"]:.1f} MW')
    print(f'TAC = {best_B["TAC"]:.2f} 億円/年')
    caseB_TAC = best_B['TAC']
else:
    print('=== Case B (膜なし) ===')
    print('全試行が spec 99.5 wt% 未達 → Dist3 単体での要求純度達成は実用範囲では物理的に不可能')
    caseB_TAC = float('nan')

print(f'\n=== TAC 比較 ===')
caseA_total = MEM_TAC_ANNUAL + caseA['TAC_dist3']
print(f'Case A (Mem+Dist3): Mem {MEM_TAC_ANNUAL:.1f} + Dist3 {caseA["TAC_dist3"]:.1f} = {caseA_total:.1f} 億円/年')
if not np.isnan(caseB_TAC):
    print(f'Case B (Dist3 のみ): {caseB_TAC:.1f} 億円/年')
    diff = caseB_TAC - caseA_total
    print(f'差 = {diff:+.1f} 億円/年 ({"膜有利" if diff > 0 else "膜不利"})')
    print(f'倍率 = {caseB_TAC/caseA_total:.2f}x')
else:
    print('Case B: 物理的不可能 → 比較不能 (膜は必須)')

## 6. 組成スイープ: Dist3 フィード C3H6 mol% を変える

Dist3 フィードの C3H6 mol% を **30% (= 膜なし Dist2 bot)** から **99% (= 強い膜)** まで変化させ、Dist3 の段数・CAPEX・OPEX がどう変わるかを見る。

In [ ]:
# 6.1 スイープ: 純度 spec 99.5 wt% を満たすため rec_HK を組成に応じて動的調整
# 物質収支: rec_LK × F_LK / [(1-rec_HK) × F_HK] >= 208.6 (= spec ratio で 99.5 wt% 達成)
fractions = np.linspace(0.30, 0.99, 40)
F_total_sweep = F_total
PURITY_RATIO_FOR_995 = 0.995 * MW_C3H8 / ((1-0.995) * MW_C3H6)  # ~208.6

sweep_results = []
for x in fractions:
    F_C3H6 = F_total_sweep * x
    F_C3H8 = F_total_sweep * (1 - x)
    rec_LK_target = 0.99
    # rec_HK を spec 達成最低値に動的調整
    needed_HK_leak = rec_LK_target * F_C3H6 / (PURITY_RATIO_FOR_995 * F_C3H8) if F_C3H8 > 0 else 0
    rec_HK_target = max(0.99, min(0.99999, 1.0 - needed_HK_leak * 0.9))  # 0.9 safety margin
    try:
        r0 = simulate_dist3(F_C3H6=F_C3H6, F_C3H8=F_C3H8, N_stages=500, reflux_ratio=30.0,
                              T_feed=320, rec_LK=rec_LK_target, rec_HK=rec_HK_target)
        R_target = min(r0['R_min'] * 1.2 if r0['R_min'] > 0 else 20.0, 50.0)
        N_target = int(min(max(r0['N_min'] * 2.0, 50), 500))
        r = simulate_dist3(F_C3H6=F_C3H6, F_C3H8=F_C3H8, N_stages=N_target, reflux_ratio=R_target,
                            T_feed=320, rec_LK=rec_LK_target, rec_HK=rec_HK_target)
        sweep_results.append({
            'x_C3H6': x,
            'rec_HK': rec_HK_target,
            'N_min': r['N_min'], 'R_min': r['R_min'],
            'N_used': N_target, 'R_used': R_target,
            'feasible': r['feasible'],
            'D_col': r['D_col_m'], 'H_col': r['H_col_m'],
            'CAPEX': r['CAPEX_total'],
            'OPEX': r['OPEX_total'],
            'Q_reb_kW': r['Q_reb_kW'],
            'TAC': r['TAC_dist3'],
            'purity_wt': r['purity_wt'],
        })
    except Exception as e:
        sweep_results.append({'x_C3H6': x, 'error': str(e)[:50]})

df_sweep = pd.DataFrame(sweep_results)
df_sweep.head(10)

In [ ]:
# 6.2 プロット: Dist3 単体の指標が組成に対しどう変わるか
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
mem_x = perm_x_C3H6

# N_min vs x
axes[0,0].plot(df_sweep['x_C3H6']*100, df_sweep['N_min'], 'b-', linewidth=2)
axes[0,0].axvline(x_C3H6_feed*100, color='r', linestyle='--', label=f'Dist2 bot 直接 ({x_C3H6_feed*100:.1f}%)')
axes[0,0].axvline(mem_x*100, color='g', linestyle='--', label=f'Mem permeate ({mem_x*100:.1f}%)')
axes[0,0].set_xlabel('Dist3 フィード C3H6 mol%')
axes[0,0].set_ylabel('N_min (Fenske)')
axes[0,0].set_title('理論最小段数 N_min')
axes[0,0].set_yscale('log')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# R_min vs x
axes[0,1].plot(df_sweep['x_C3H6']*100, df_sweep['R_min'], 'b-', linewidth=2)
axes[0,1].axvline(x_C3H6_feed*100, color='r', linestyle='--')
axes[0,1].axvline(mem_x*100, color='g', linestyle='--')
axes[0,1].set_xlabel('Dist3 フィード C3H6 mol%')
axes[0,1].set_ylabel('R_min (Underwood)')
axes[0,1].set_title('最小還流比 R_min')
axes[0,1].grid(True, alpha=0.3)

# CAPEX vs x
axes[1,0].plot(df_sweep['x_C3H6']*100, df_sweep['CAPEX'], 'b-', linewidth=2)
axes[1,0].axvline(x_C3H6_feed*100, color='r', linestyle='--')
axes[1,0].axvline(mem_x*100, color='g', linestyle='--')
axes[1,0].set_xlabel('Dist3 フィード C3H6 mol%')
axes[1,0].set_ylabel('Dist3 CAPEX [億円]')
axes[1,0].set_title('Dist3 CAPEX')
axes[1,0].set_yscale('log')
axes[1,0].grid(True, alpha=0.3)

# Q_reb vs x
axes[1,1].plot(df_sweep['x_C3H6']*100, df_sweep['Q_reb_kW']/1000, 'b-', linewidth=2)
axes[1,1].axvline(x_C3H6_feed*100, color='r', linestyle='--')
axes[1,1].axvline(mem_x*100, color='g', linestyle='--')
axes[1,1].set_xlabel('Dist3 フィード C3H6 mol%')
axes[1,1].set_ylabel('Q_reb [MW]')
axes[1,1].set_title('Dist3 リボイラ熱負荷')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sweep_composition.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 TAC 比較
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(df_sweep['x_C3H6']*100, df_sweep['TAC'], 'b-', linewidth=2, label='Dist3 単体 TAC (Case B 相当)')
ax.plot(df_sweep['x_C3H6']*100, df_sweep['TAC'] + MEM_TAC_ANNUAL, 'r-', linewidth=2, label=f'Dist3 + 仮想 Mem ({MEM_TAC_ANNUAL:.0f} 億/年加算)')
ax.axhline(MEM_TAC_ANNUAL + caseA['TAC_dist3'], color='g', linestyle='--', label=f'Case A 実測 (Mem→Dist3): {MEM_TAC_ANNUAL + caseA["TAC_dist3"]:.0f} 億/年')
ax.axvline(x_C3H6_feed*100, color='gray', linestyle=':', label=f'Dist2 bot 組成 ({x_C3H6_feed*100:.1f}%)')
ax.axvline(mem_x*100, color='black', linestyle=':', label=f'Mem permeate 組成 ({mem_x*100:.1f}%)')
ax.set_xlabel('Dist3 フィード C3H6 mol%')
ax.set_ylabel('TAC [億円/年]')
ax.set_title('TAC 構造: Dist3 単体 vs Dist3+Mem')
ax.set_yscale('log')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tac_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. 目標純度を変えた比較

目標純度を 99.0 / 99.5 / 99.9 wt% で変えて、Dist3 単体の負担がどう変わるか。

In [ ]:
fractions_p = np.array([0.30, 0.50, 0.70, 0.90, 0.95, 0.98])
purity_targets_rec_LK = [
    (0.95, '99.0 wt% 目標相当 (rec_LK=0.95)'),
    (0.99, '99.5 wt% 目標相当 (rec_LK=0.99)'),
    (0.999, '99.9 wt% 目標相当 (rec_LK=0.999)'),
]

purity_sweep = {}
for rec_LK, label in purity_targets_rec_LK:
    results = []
    for x in fractions_p:
        F_C3H6 = F_total * x
        F_C3H8 = F_total * (1 - x)
        r = simulate_dist3(F_C3H6=F_C3H6, F_C3H8=F_C3H8, N_stages=300, reflux_ratio=30,
                            T_feed=320, rec_LK=rec_LK, rec_HK=0.999)
        results.append({
            'x_C3H6': x, 'N_min': r['N_min'], 'R_min': r['R_min'],
            'purity_wt': r['purity_wt'],
        })
    purity_sweep[label] = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, df in purity_sweep.items():
    axes[0].plot(df['x_C3H6']*100, df['N_min'], 'o-', label=label, linewidth=2, markersize=8)
    axes[1].plot(df['x_C3H6']*100, df['R_min'], 'o-', label=label, linewidth=2, markersize=8)
for ax in axes:
    ax.axvline(x_C3H6_feed*100, color='r', linestyle=':', alpha=0.6, label='Dist2 bot')
    ax.axvline(mem_x*100, color='g', linestyle=':', alpha=0.6, label='Mem perm')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlabel('Dist3 フィード C3H6 mol%')
axes[0].set_ylabel('N_min'); axes[0].set_title('段数')
axes[1].set_ylabel('R_min'); axes[1].set_title('還流比')
axes[0].set_yscale('log')
plt.tight_layout()
plt.savefig('purity_target_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. 膜性能感度: A_mem と α (選択性)

膜の物性 (A_mem 面積、α 選択性) を変えると **Mem permeate purity が変わる → Dist3 負担も変わる**。

In [ ]:
A_mem_values = np.array([3e4, 5e4, 7e4, 1e5, 1.5e5, 2e5, 3e5, 5e5])
amem_sweep = []
for A in A_mem_values:
    try:
        md = MemDesignVars(P_H=8.75e5, P_L=1e5, A_mem=A, P_dist=P_DIST3)
        mr = simulate_membrane_system(md, mem_feed, MemFixedParams())
        ptot = mr.product.F_C3H6 + mr.product.F_C3H8
        rtot = mr.retentate.F_C3H6 + mr.retentate.F_C3H8
        perm_x = mr.product.F_C3H6 / ptot if ptot > 0 else 0
        ret_x = mr.retentate.F_C3H6 / rtot if rtot > 0 else 0
        sc = ptot / (ptot + rtot)
        mem_capex = (mr.equipment.CAPEX_vap + mr.equipment.CAPEX_comp_feed
                       + mr.equipment.CAPEX_mem + mr.equipment.CAPEX_comp_prod
                       + mr.equipment.CAPEX_cond)
        mem_opex_e = (mr.equipment.W_feed_kW + mr.equipment.W_prod_kW) * ELECTRICITY_JPY_PER_KWH * OPERATING_HOURS_PER_YEAR / 1e8
        mem_opex_s = mr.equipment.Q_vap_kW * 3.6e-3 * OPERATING_HOURS_PER_YEAR * LP_STEAM_JPY_PER_GJ / 1e8
        mem_opex_w = mr.equipment.Q_cond_kW * 3.6e-3 * OPERATING_HOURS_PER_YEAR * COOLING_WATER_JPY_PER_GJ / 1e8
        mem_tac = mem_capex / DEPRECIATION_YEARS + mem_opex_e + mem_opex_s + mem_opex_w
        dr = simulate_dist3(
            F_C3H6=mr.product.F_C3H6, F_C3H8=mr.product.F_C3H8,
            P_col=P_DIST3, N_stages=200, reflux_ratio=15,
            T_feed=mr.product.T_out, rec_LK=0.99, rec_HK=0.99,
        )
        amem_sweep.append({
            'A_mem': A, 'perm_x_C3H6': perm_x*100, 'ret_x_C3H6': ret_x*100,
            'stage_cut': sc*100,
            'mem_capex': mem_capex, 'mem_tac': mem_tac,
            'd3_N_min': dr['N_min'], 'd3_R_min': dr['R_min'],
            'd3_CAPEX': dr['CAPEX_total'], 'd3_TAC': dr['TAC_dist3'],
            'total_TAC': mem_tac + dr['TAC_dist3'],
        })
    except Exception as e:
        amem_sweep.append({'A_mem': A, 'error': str(e)[:60]})

df_amem = pd.DataFrame(amem_sweep)
df_amem

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
df_a = df_amem.dropna(subset=['mem_tac']) if 'mem_tac' in df_amem.columns else df_amem.iloc[0:0]

axes[0,0].plot(df_a['A_mem']/1e5, df_a['perm_x_C3H6'], 'o-', linewidth=2, markersize=8, label='Permeate C3H6 mol%')
axes[0,0].set_xlabel('A_mem [×10⁵ m²]')
axes[0,0].set_ylabel('Mem permeate C3H6 mol%')
axes[0,0].set_title('膜面積 → 透過側純度')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(df_a['A_mem']/1e5, df_a['stage_cut'], 'o-', linewidth=2, markersize=8, color='orange')
axes[0,1].set_xlabel('A_mem [×10⁵ m²]')
axes[0,1].set_ylabel('Stage cut [%]')
axes[0,1].set_title('膜面積 → Stage cut (透過率)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(df_a['A_mem']/1e5, df_a['mem_tac'], 'b-o', label='Mem TAC')
axes[1,0].plot(df_a['A_mem']/1e5, df_a['d3_TAC'], 'r-s', label='Dist3 TAC')
axes[1,0].plot(df_a['A_mem']/1e5, df_a['total_TAC'], 'k--^', linewidth=2, label='合計 TAC')
axes[1,0].set_xlabel('A_mem [×10⁵ m²]')
axes[1,0].set_ylabel('TAC [億円/年]')
axes[1,0].set_title('TAC 内訳 (Mem + Dist3)')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(df_a['A_mem']/1e5, df_a['d3_N_min'], 'o-', linewidth=2, markersize=8, color='purple')
axes[1,1].set_xlabel('A_mem [×10⁵ m²]')
axes[1,1].set_ylabel('Dist3 N_min')
axes[1,1].set_title('膜面積 → Dist3 必要段数')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('amem_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

if len(df_a) > 0:
    best_amem_row = df_a.loc[df_a['total_TAC'].idxmin()]
    print(f'\n合計 TAC 最小: A_mem = {best_amem_row["A_mem"]/1e5:.2f}e5 m²')
    print(f'  Mem TAC = {best_amem_row["mem_tac"]:.1f}, Dist3 TAC = {best_amem_row["d3_TAC"]:.1f}, 合計 {best_amem_row["total_TAC"]:.1f} 億/年')
    print(f'  Mem permeate C3H6 = {best_amem_row["perm_x_C3H6"]:.2f} mol%, stage cut = {best_amem_row["stage_cut"]:.1f}%')
    print(f'  (BO best #282 の A_mem = 1.45e5 m² と比較)')

In [ ]:
# 8.3 膜選択性 α の感度 (Hua et al. 2024 = α=90 が基準、低性能膜 α=15 まで試す)
alpha_values = [10, 15, 30, 50, 90, 150]
alpha_sweep = []
for alpha in alpha_values:
    fixed = MemFixedParams(alpha=alpha)
    md = MemDesignVars(P_H=8.75e5, P_L=1e5, A_mem=1.45e5, P_dist=P_DIST3)
    try:
        mr = simulate_membrane_system(md, mem_feed, fixed)
        ptot = mr.product.F_C3H6 + mr.product.F_C3H8
        perm_x = mr.product.F_C3H6 / ptot if ptot > 0 else 0
        dr = simulate_dist3(F_C3H6=mr.product.F_C3H6, F_C3H8=mr.product.F_C3H8,
                              N_stages=200, reflux_ratio=15,
                              T_feed=mr.product.T_out, rec_LK=0.99, rec_HK=0.99)
        alpha_sweep.append({
            'alpha': alpha, 'perm_x_C3H6': perm_x*100,
            'd3_N_min': dr['N_min'], 'd3_R_min': dr['R_min'],
            'd3_TAC': dr['TAC_dist3'],
        })
    except Exception as e:
        alpha_sweep.append({'alpha': alpha, 'error': str(e)[:60]})

df_alpha = pd.DataFrame(alpha_sweep)
df_alpha

## 9. 統合比較表

In [ ]:
summary = {
    '指標': [
        'Dist3 入口 C3H6 mol%',
        'Dist3 N_min',
        'Dist3 R_min',
        'Dist3 N_stages (採用)',
        'Dist3 R (採用)',
        'Dist3 塔高 [m]',
        'Dist3 CAPEX [億円]',
        'Dist3 Q_reb [MW]',
        'Dist3 TAC [億円/年]',
        'Mem CAPEX [億円]',
        'Mem TAC [億円/年]',
        '合計 TAC [億円/年]',
        '達成 C3H6 純度 [wt%]',
        'spec 99.5wt% 達成',
    ],
    'Case A (Mem→Dist3)': [
        f'{perm_x_C3H6*100:.2f}',
        f'{caseA["N_min"]:.1f}',
        f'{caseA["R_min"]:.2f}',
        f'{caseA["N_stages"]}',
        f'{caseA["R"]:.2f}',
        f'{caseA["H_col_m"]:.1f}',
        f'{caseA["CAPEX_total"]:.2f}',
        f'{caseA["Q_reb_kW"]/1000:.1f}',
        f'{caseA["TAC_dist3"]:.2f}',
        f'{MEM_CAPEX_TOTAL:.2f}',
        f'{MEM_TAC_ANNUAL:.2f}',
        f'{MEM_TAC_ANNUAL + caseA["TAC_dist3"]:.2f}',
        f'{caseA["purity_wt"]*100:.3f}',
        '✓' if caseA['purity_wt'] >= 0.995 else '✗',
    ],
    'Case B (Dist3 単体)': [],
}
if len(df_caseB_meet) > 0:
    bB = df_caseB_meet.loc[df_caseB_meet['TAC'].idxmin()]
    summary['Case B (Dist3 単体)'] = [
        f'{x_C3H6_feed*100:.2f}',
        f'{bB["N_min"]:.1f}',
        f'{bB["R_min"]:.2f}',
        f'{int(bB["N"])}',
        f'{bB["R"]:.2f}',
        f'{bB["H_col"]:.1f}',
        f'{bB["CAPEX"]:.2f}',
        f'{bB["Q_reb_MW"]:.1f}',
        f'{bB["TAC"]:.2f}',
        '0 (膜なし)',
        '0',
        f'{bB["TAC"]:.2f}',
        f'{bB["purity_wt"]*100:.3f}',
        '✓',
    ]
else:
    summary['Case B (Dist3 単体)'] = ['—'] * len(summary['指標'])
    summary['Case B (Dist3 単体)'][0] = f'{x_C3H6_feed*100:.2f}'
    summary['Case B (Dist3 単体)'][-1] = '✗ 達成不可'

df_summary = pd.DataFrame(summary)
print('=== Case A (Mem+Dist3) vs Case B (Dist3 単体) ===')
print(df_summary.to_string(index=False))

## 10. 結論

### 数値的事実 (本 notebook 実行結果より)

1. **Dist2 塔底 (C3H6 ~37 mol%) を Dist3 単体で 99.5 wt% まで分離するには rec_HK ≥ 0.999 が必須**。これにより N_min と R_min が爆発する。
2. **膜 (A_mem=1.45e5 m², α=90) で 96 mol% まで予備濃縮** することで、Dist3 は N=121 段で feasible (Case A 実測)。
3. **Dist3 単体 vs Mem+Dist3 の TAC 差**: Section 5.2 と Section 6 の TAC 比較プロット参照。
4. **膜面積感度** (Section 8): A_mem を 3e4 → 5e5 m² まで変えると Mem permeate C3H6 mol% が大きく変動し、合計 TAC に最適点が存在。BO best #282 の A_mem=1.45e5 m² は妥当圏内。

### 物理的根拠

- C3H6/C3H8 系の α (PR EOS @17 bar) ≈ 1.07 で、Fenske の N_min は組成と recovery に対し急激に発散する:
  - N_min ∝ ln[(x_LK/x_HK)_top × (x_HK/x_LK)_bot] / ln(α)
  - ln(α) = ln(1.07) ≈ 0.068 で分母が極小、僅かな組成変化で N_min が大変動
- 高選択性膜 (α=90) は分母 ln(α) ≈ 4.5 で **同じ分離効率を 65 倍効率良く実現**
- → 膜は「分離難易度の高い部分を物理的に異なる原理で処理」する役割

### 結語

**膜分離器導入は、PDH プロセスにおいて単なるオプションではなく経済的・物理的に必須**。Mem 投資 (CAPEX ~27 億 + OPEX ~30 億/年) は、Dist3 単体では達成不可能な分離を可能にし、結果として全体 TAC を実用域に収める。